# radcoolpv — radiative cooling of silicon PV

Edit the YAML, run the cell, read the numbers. Nothing is installed on your own
machine and nothing here needs a compiled electromagnetic solver.

**Run all** takes about a minute and produces:

* the **temperatures** a module settles at, and the **powers** that balance there;
* the **PV parameters** — short-circuit current, open-circuit voltage, fill
  factor, maximum power, efficiency, and the temperature coefficient;
* the figures for each run.

The worked example is Akerboom *et al.*, *ACS Photonics* **9**, 3831–3840 (2022),
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389) —
silica microcylinders on a silicon module. It is an *example*: the wavelength
range, the materials, the geometry and the layer stack are all yours to change.

## 1. Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Image, Markdown, display

PROJECT = Path("/content/radcoolpv-py")
REPO    = "https://github.com/gsilvaoelker/radcoolpv-py.git"

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO,
                    str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline
print("radcoolpv ready in", PROJECT)

## 2. A helper that prints the results

Every run writes a `run.json` next to its figures. It holds the resolved
configuration, hashes of every input, and the scalar results. This cell just
formats it for reading; look at the file itself once, it is the honest record of
what was computed.

Quantities the run did not actually solve for are **absent** rather than zero.
A cooling-curve run has no operating point, so it reports no efficiency — as
opposed to reporting an efficiency of zero.

In [ ]:
UNITS = [("_W_per_m2", "W/m2"), ("_A_per_m2", "A/m2"), ("_perc_per_K", "%/K"),
         ("_K", "K"), ("_V", "V")]

def _label(key):
    for suffix, unit in UNITS:
        if key.endswith(suffix):
            return key[: -len(suffix)].replace("_", " "), unit
    return key.replace("_", " "), ""

def show(ctx, figures=True):
    """Print the temperatures, powers and PV parameters, then the figures."""
    record = json.load(open(Path(ctx.results_dir) / "run.json"))
    optics, results = record["optics"], record.get("thermal_results", {})

    lo, hi = optics["wavelength_range_um"]
    note = " (silicon absorptance inferred from the supplied emittance)" \
        if optics["silicon_from_emittance"] else ""
    print(f"wavelength {lo}-{hi} um, {optics['n_lambda']} points, "
          f"{optics['angles']}{note}\n")

    for key, value in results.items():
        if value is None:
            continue
        name, unit = _label(key)
        print(f"  {name:<42} {value:>12.4f}  {unit}")

    band = record.get("band_averages_percent")
    if band:
        print()
        for key, value in band.items():
            print(f"  {key.replace('_', ' '):<42} {value:>12.4f}  %")

    if figures:
        for png in sorted((Path(ctx.results_dir) / "figures").glob("*.png")):
            display(Image(filename=str(png)))

## 3. The validation — no solver needed

The paper measured the emittance of three surfaces: a bare Au/Si module, the
same module under a flat silica slab, and the slab patterned into the
microcylinder array. Those measured spectra are digitized in the repository, so
this group runs the **thermal model alone** and can be checked against the
paper's Figure 5b without computing any optics.

That separation is the point: any disagreement here is the thermal model's, not
the solver's.

In [ ]:
cases = {c.case_name: c for c in config.load_cases("validation/akerboom.yaml")}
print("\n".join(cases))

### 3a. With the convection coefficient the paper states

The paper's Methods give $h = 6$ W/m²/K. Run the three surfaces with it.

In [ ]:
contexts = {}
for name in ("B1_cooling_h6_bare", "B2_cooling_h6_flat_silica", "B3_cooling_h6_cylinders"):
    contexts[name] = pipeline.run(cases[name])

rows = "\n".join(
    f"| `{n}` | {contexts[n].thermal.equil_temp:.1f} K | {t} K |"
    for n, t in zip(contexts, (360, 339, 336)))
display(Markdown("| Case | radcoolpv | Paper Fig. 5b |\n|---|---:|---:|\n" + rows))

The calculated temperatures are far above the ones the paper plots. This is not
a bug to hide — it is the result. Read section 5 below before drawing a
conclusion from it.

### 3b. With a convection coefficient fitted to the paper's own figure

A single least-squares fit to all three digitized Figure 5b curves gives
$h = 12.54$ W/m²/K. That reproduces the paper's temperatures — but it is a
**calibration**, not an independent confirmation.

In [ ]:
for name in ("B4_cooling_fitted_bare", "B5_cooling_fitted_flat_silica",
             "B6_cooling_fitted_cylinders"):
    contexts[name] = pipeline.run(cases[name])

rows = "\n".join(
    f"| `{n}` | {contexts[n].thermal.equil_temp:.1f} K | {t} K |"
    for n, t in zip(list(contexts)[3:], (360, 339, 336)))
display(Markdown("| Case | radcoolpv | Paper Fig. 5b |\n|---|---:|---:|\n" + rows))

### 3c. The full record for one case

Temperatures, the energy balance, and the figures.

In [ ]:
show(contexts["B6_cooling_fitted_cylinders"])

The energy balance closes on itself:

$$P_\mathrm{rad}-P_\mathrm{atm}+P_\mathrm{conv}-P_\mathrm{sun}=P_\mathrm{cool}=0$$

at the equilibrium temperature. Add up the four powers printed above and check.

## 4. PV parameters, still with no solver

The digitized spectra above stop at 2 µm, well above silicon's band gap at about
1.1 µm, so they say nothing about sunlight and cannot drive a cell. For the
electrical result you need optical data that reaches into the solar band.

This example reads a committed free-form structure that does, and runs the whole
chain — optics, energy balance, and the single-diode cell — with no solver.

In [ ]:
pv = pipeline.run(config.load_cases("examples/freeform_pv.yaml")[0])
show(pv)

## 5. Edit this and run your own case

Everything below is yours. Change the wavelength range, the layer thicknesses,
the materials, the geometry, the ambient temperature, the convection
coefficient. Run the cell to write the file, then the next one to execute it.

A few things worth knowing:

* **The wavelength range must fit inside every material's table.** The error
  names the file that is too narrow. `RII_Olmon_2012_ev_Au` stops at 24.93 µm,
  which is what sets the upper limit in the example below.
* **Commenting a key out reverts it to the code default, which is often the
  opposite of what you wanted.** `# thermal: false` turns thermal *on*. Set
  values explicitly. Commenting out a list item, such as a `structure` layer, is
  safe.
* **Wavelength resolution is not free.** `n` controls how well the integrals are
  resolved; a coarse grid runs fast and answers a different question.

In [ ]:
%%writefile my_case.yaml
# Cooling curve for a surface whose emittance you supply.
# Change anything here, then run the next cell.

run:
  optics: false               # no solver: read the spectrum from a file
  thermal: true
  plots: true
  mode: cooling_curve
  write_outputs: true
  results_dir: results/my_case
  optics_results: validation/data/fig5a_measured_emittance.txt
  optics_results_angles: hemispherical
  optics_results_emittance_column: 3     # 1 bare, 2 flat silica, 3 cylinders

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 281}
  angles: hemispherical

thermal:
  ambient_temperature: 300.0       # K
  convection_coefficient: 12.54    # W/m2-K, everything non-radiative
  absorbed_solar_power: 808.0      # W/m2, absorbed -- not the incident irradiance
  equilibrium: auto
  cooling_temperature: {min: 260.0, max: 380.0, n: 121}

In [ ]:
show(pipeline.run(config.load_cases("my_case.yaml")[0]))

## 6. Use your own optical data

Upload a text file with wavelength in the first column and one or more spectra
in the others — measured emittance, a spectrum from another program, whatever
you have. Wavelength is in micrometres.

Point `optics_results` at the uploaded filename and `optics_results_emittance_column`
at the column you want, then re-run section 5.

**If your file reaches below about 1.1 µm you also get the PV parameters.**
Above the band gap essentially everything absorbed is absorbed in the silicon,
so radcoolpv takes the silicon absorptance to equal the emittance there and zero
below. `run.json` records that this was assumed rather than solved.

You can also drop a material here: a CSV named `<Model>.csv` with columns
`lambda_um,n,k` copied into `radcoolpv/materials/data/` becomes usable as
`<Model>` in the `materials:` block straight away.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    print(f"{name}: {len(uploaded[name])} bytes")
    if name.lower().endswith(".csv") and "," in uploaded[name].decode(errors="ignore")[:200]:
        target = PROJECT / "radcoolpv" / "materials" / "data" / name
        target.write_bytes(uploaded[name])
        print(f"  installed as material {Path(name).stem!r}")

## 7. Optional — compile the electromagnetic solver

Everything above ran without S4. To compute the optics from a geometry rather
than read them from a file, S4 has to be built from source: it is a C++
extension with no PyPI package. Expect about ten minutes.

Skip this unless you want to change the *structure* rather than the spectrum.

In [ ]:
RUN_S4_BUILD = False        # set True to build

S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"   # the tested revision

def build_s4():
    import importlib, importlib.util
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable."); return
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential", "git",
                    "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
                    "libopenblas-dev", "libsuitesparse-dev"], check=True)
    src = Path("/content/S4")
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/phoebe-p/S4.git", str(src)], check=True)
    subprocess.run(["git", "checkout", S4_COMMIT], cwd=src, check=True)
    subprocess.run(["make", "-j2", "S4_pyext"], cwd=src, check=True)
    importlib.invalidate_caches()
    import S4; print("S4:", S4.__file__)

if RUN_S4_BUILD:
    build_s4()
else:
    print("Skipped. Sections 1-6 do not need S4.")

## 8. Optional — compute the optics from the geometry

This is where the microcylinders stop being a spectrum on disk and become a
structure. The cell below estimates the cost before it starts, because the
converged settings in `validation/akerboom.yaml` are hours of computation and a
free Colab runtime can be reclaimed underneath you.

The reduced grid it runs by default is a **smoke test**: enough to see the
machinery work, not enough to quote. Raise `n` and `s4_modes` and re-check that
the number you care about has stopped moving before you report it.

In [ ]:
from radcoolpv.optics import s4_backend

def cost(cfg, seconds_per_solve=0.24):
    n_lambda = cfg.simulation.wavelength.n
    n_dir = len(cfg.direction_arrays()[0])
    n_pol = len(cfg.simulation.polarization_names())
    solves = n_lambda * n_dir * n_pol
    print(f"{n_lambda} wavelengths x {n_dir} direction(s) x {n_pol} polarization(s) "
          f"= {solves} solves, roughly {solves * seconds_per_solve / 60:.0f} min")
    return solves

cfg = cases["A3_optics_cylinders"]
cfg.simulation.wavelength.n = 60      # converged value is 281
cfg.simulation.s4_modes = 20          # converged value is 60
cfg.run.results_dir = "results/smoke_test"

cost(cfg)
if s4_backend.is_available():
    show(pipeline.run(cfg))
else:
    print("\nS4 is not built in this runtime — run section 7 first.")

## 9. Try these

1. In section 5, sweep the convection coefficient from 6 to 15 W/m²/K and plot
   the equilibrium temperature against it. How much of the paper's claimed
   cooling survives if $h$ is uncertain by a factor of two?
2. Change `optics_results_emittance_column` from 3 to 1. That is the bare
   module. How much of the temperature drop comes from the silica at all?
3. In section 8, raise `s4_modes` from 20 through 40 to 60 at fixed `n`. Where
   does the 8–13 µm emittance stop moving? That is your converged mode count,
   and nothing below it is quotable.
4. Set a wavelength range that excludes the 8–13 µm atmospheric window entirely
   and re-run section 5. Explain the temperature you get.

## What this validation does and does not show

The optics agree with the paper: the calculated 7.5–16 µm emittance comes out
0.032, 0.842 and 0.984 for the three surfaces, against 0.036, 0.843 and 0.977
digitized from Figure 3a.

The thermal model does not agree at the paper's own stated $h = 6$ W/m²/K, and
the reason is checkable rather than mysterious. Put a perfect non-emitter under
that balance — no emission, no sunlight absorbed beyond the stated 808 W/m² —
and the temperature must be $300 + 808/6 = 434.7$ K. The paper's figure implies
something near 366.5 K instead, which no emitter can produce at that
coefficient. A fitted $h = 12.54$ W/m²/K reproduces Figure 5b for all three
surfaces at once, which is what `B4`–`B6` above show.

So: treat the fitted agreement as a calibration, and do not cite it as an
independent validation of the thermal model.